# Laboratorio #5 — Modelos de lenguaje
**Natural Language Processing — UFM**

In [ ]:
import random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

import kagglehub, os
path = kagglehub.dataset_download('manuelmaaf97/quijote')
print('Path:', path)
print('Archivos:', os.listdir(path))

plt.rcParams["figure.dpi"] = 100

Path: /Users/georgealbadr/.cache/kagglehub/datasets/manuelmaaf97/quijote/versions/1
Archivos: ['don-quijote.txt']


## 1. Preparación del corpus para modelado de secuencias

Un modelo de lenguaje estima P(siguiente palabra | palabras anteriores), así que necesita el texto
como **secuencias ordenadas**, no como bolsa de palabras. Por eso el pipeline es más simple que en
los labs anteriores: solo segmentar en oraciones y tokenizar, sin normalizar nada más.

In [ ]:
archivo = os.path.join(path, "don-quijote.txt")
with open(archivo, encoding="utf-8") as f:
    crudo = f.read()

print(f"archivo crudo: {len(crudo):,} caracteres")


ini = crudo.index("El ingenioso hidalgo don Quijote de la Mancha")
fin = crudo.rindex("Fin") + len("Fin")
texto_quijote = crudo[ini:fin].strip()

print(f"texto limpio : {len(texto_quijote):,} caracteres "
      f"({len(crudo) - len(texto_quijote):,} de boilerplate removidos)\n")
print(texto_quijote[:250])
print("   ...")
print(texto_quijote[-120:])

archivo crudo: 2,117,498 caracteres
texto limpio : 2,097,952 caracteres (19,546 de boilerplate removidos)

El ingenioso hidalgo don Quijote de la Mancha


TASA

Yo, Juan Gallo de Andrada, escribano de Cámara del Rey nuestro señor, de
los que residen en su Consejo, certifico y doy fe que, habiendo visto por
los señores dél un libro intitulado El ingenioso 
   ...
rías, que, por las de mi verdadero don
Quijote, van ya tropezando, y han de caer del todo, sin duda alguna. Vale.



Fin


In [ ]:
oraciones = sent_tokenize(texto_quijote, language="spanish")

print(f"oraciones detectadas: {len(oraciones):,}")
print("\nprimeras 3:")
for o in oraciones[:3]:
    print(" -", o[:100])

oraciones detectadas: 9,549

primeras 3:
 - El ingenioso hidalgo don Quijote de la Mancha


TASA

Yo, Juan Gallo de Andrada, escribano de Cámara
 - Y, para que dello conste, di la
presente en Valladolid, a veinte días del mes de deciembre de mil y

 - Juan Gallo de Andrada.


In [ ]:
from pprint import pprint
oraciones_tok = [
    ["<s>"] + word_tokenize(oracion, language="spanish") + ["</s>"]
    for oracion in oraciones
]


idx = next(i for i, o in enumerate(oraciones_tok) if 12 <= len(o) <= 25)

print(f"ejemplo — oracion {idx} ({len(oraciones_tok[idx])} tokens):")
pprint(oraciones_tok[idx], width=90, compact=True) 

print(f"\ntotal de oraciones tokenizadas: {len(oraciones_tok):,}")
print(f"tokens totales (con <s>/</s>): {sum(len(o) for o in oraciones_tok):,}")

ejemplo — oracion 12 (23 tokens):
['<s>', 'Fecha', 'en', 'Valladolid', ',', 'a', 'veinte', 'y', 'seis', 'días', 'del',
 'mes', 'de', 'setiembre', 'de', 'mil', 'y', 'seiscientos', 'y', 'cuatro', 'años', '.',
 '</s>']

total de oraciones tokenizadas: 9,549
tokens totales (con <s>/</s>): 458,083


In [ ]:
random.seed(42)

oraciones_mezcladas = random.sample(oraciones_tok, len(oraciones_tok))
n = len(oraciones_mezcladas)
train_oraciones = oraciones_mezcladas[:int(n * 0.8)]
val_oraciones   = oraciones_mezcladas[int(n * 0.8):int(n * 0.9)]
test_oraciones  = oraciones_mezcladas[int(n * 0.9):]

for nombre, conj in [("train", train_oraciones), ("val", val_oraciones), ("test", test_oraciones)]:
    print(f"{nombre:<6} {len(conj):>6} oraciones  ({len(conj)/n*100:.1f}%)")

train    7639 oraciones  (80.0%)
val       955 oraciones  (10.0%)
test      955 oraciones  (10.0%)


In [ ]:
vocab_train = Counter(tok for oracion in train_oraciones for tok in oracion)
print(f"Vocabulario de entrenamiento: {len(vocab_train):,} palabras distintas")

tokens_test = [tok for oracion in test_oraciones for tok in oracion]
oov = sum(1 for tok in tokens_test if tok not in vocab_train)
proporcion_oov = oov / len(tokens_test) if tokens_test else 0

print(f"Tokens en prueba: {len(tokens_test):,}")
print(f"Tokens OOV      : {oov:,}  ({proporcion_oov*100:.2f}%)")

Vocabulario de entrenamiento: 22,650 palabras distintas
Tokens en prueba: 44,569
Tokens OOV      : 1,471  (3.30%)


**¿Qué relación tiene el problema de "palabras nunca vistas" con la dispersión de datos?**

El vocabulario de entrenamiento es apenas una muestra del español de Cervantes, y por más
grande que sea esa muestra siempre queda una cola larga de palabras raras que aparecen una sola vez
o ninguna en entrenamiento pero sí en prueba. Esto es dispersión de datos vista desde el
vocabulario en lugar de desde una matriz:  por ejemplo en el Lab#2 la dispersión aparecía porque cada
documento activa solo una fracción minúscula de las columnas del vocabulario total, y acá pasa lo
mismo pero a nivel de oraciones completas — ninguna partición del corpus, por representativa que
sea, alcanza a cubrir el vocabulario entero, porque el idioma sigue produciendo combinaciones y
formas nuevas más rápido de lo que cualquier muestra finita puede registrarlas.

La consecuencia práctica es directa: un modelo de n-gramas nunca vio esas palabras OOV durante el
entrenamiento, así que no tiene ningún conteo asociado a ellas. Sin suavizado, cualquier oración de
prueba que contenga una sola palabra OOV recibe probabilidad cero completa, sin importar qué tan
bien estimado esté el resto de la oración. Es el mismo problema de fondo que motivó TF-IDF y el
suavizado de Laplace en Naive Bayes: una muestra finita nunca es densa donde el vocabulario es
enorme, y el modelo tiene que estar preparado para lo que no vio.

## 2. Construcción de modelos n-grama

In [ ]:
from itertools import chain

def con_padding(oracion, n):
    """Agrega los <s> extra que necesita un modelo de n-gramas."""
    return ["<s>"] * (n - 2) + oracion if n > 2 else list(oracion)


padded = [con_padding(o, 3) for o in train_oraciones]

conteo_uni = Counter(chain.from_iterable(o[2:]                 for o in padded))
conteo_bi  = Counter(chain.from_iterable(zip(o[1:], o[2:])     for o in padded))
conteo_tri = Counter(chain.from_iterable(zip(o, o[1:], o[2:])  for o in padded))

total_ctx_bi  = Counter(chain.from_iterable(o[1:-1]                for o in padded))
total_ctx_tri = Counter(chain.from_iterable(zip(o[:-1], o[1:-1])  for o in padded))

total_uni = sum(conteo_uni.values())
print(f"tokens de entrenamiento (predecibles): {total_uni:,}")
print(f"contextos distintos de bigrama : {len(total_ctx_bi):,}")
print(f"contextos distintos de trigrama: {len(total_ctx_tri):,}")

siguientes_don = Counter({w: c for (ctx, w), c in conteo_bi.items() if ctx == "don"})
print(f"\nejemplo — palabras mas probables despues de 'don':")
print(siguientes_don.most_common(5))

tokens de entrenamiento (predecibles): 362,340
contextos distintos de bigrama : 22,649
contextos distintos de trigrama: 129,305

ejemplo — palabras mas probables despues de 'don':
[('Quijote', 1279), ('Quijote-', 436), ('Fernando', 105), ('Antonio', 43), ('Luis', 28)]


In [ ]:
def p_unigrama(palabra):
    return conteo_uni[palabra] / total_uni


def p_bigrama(palabra, anterior):
    total = total_ctx_bi[anterior]
    return conteo_bi[(anterior, palabra)] / total if total else 0.0


def p_trigrama(palabra, ant2, ant1):
    total = total_ctx_tri[(ant2, ant1)]
    return conteo_tri[(ant2, ant1, palabra)] / total if total else 0.0


print(f"P('Quijote')              = {p_unigrama('Quijote'):.6f}")
print(f"P('Quijote' | 'don')      = {p_bigrama('Quijote', 'don'):.6f}")
print(f"P('Quijote' | 'el','don') = {p_trigrama('Quijote', 'el', 'don'):.6f}")

antes = (len(conteo_bi), len(conteo_tri))
p_bigrama("x", "contexto_inexistente"); p_trigrama("x", "jamas", "visto")
print(f"\nn-gramas antes {antes} / despues {(len(conteo_bi), len(conteo_tri))} -> no se ensucian")

P('Quijote')              = 0.003715
P('Quijote' | 'don')      = 0.606449
P('Quijote' | 'el','don') = 0.187500

n-gramas antes (129312, 254662) / despues (129312, 254662) -> no se ensucian


In [ ]:
import math

def prob_oracion(oracion, modelo):
    n = {"uni": 1, "bi": 2, "tri": 3}[modelo]
    o = con_padding(oracion, n)
    inicio = n - 1 if n > 1 else 1

    log_p = 0.0
    for i in range(inicio, len(o)):
        if modelo == "uni":
            p = p_unigrama(o[i])
        elif modelo == "bi":
            p = p_bigrama(o[i], o[i - 1])
        else:
            p = p_trigrama(o[i], o[i - 2], o[i - 1])

        if p == 0.0:
            return 0.0, float("-inf")

        log_p += math.log(p)

    return math.exp(log_p), log_p


ejemplo = next(o for o in val_oraciones if 8 <= len(o) <= 14)
print("Oracion de ejemplo:")
print(" ", " ".join(ejemplo), "\n")

for modelo in ["uni", "bi", "tri"]:
    p, log_p = prob_oracion(ejemplo, modelo)
    print(f"{modelo:<4} P(oracion) = {p:.3e}   log P = {log_p:.3f}")

Oracion de ejemplo:
  <s> ¿Qué es esto que oigo ? </s> 

uni  P(oracion) = 5.746e-20   log P = -44.303
bi   P(oracion) = 0.000e+00   log P = -inf
tri  P(oracion) = 0.000e+00   log P = -inf


**¿Cómo permite el supuesto de Markov pasar de la regla de la cadena a los modelos n-grama?**

El supuesto de Markov permite simplificar la regla de la cadena estableciendo que la probabilidad
de que aparezca una palabra depende únicamente de un número fijo de palabras anteriores.

La regla de la cadena plantea dos problemas. El primero es que, a medida que la secuencia se hace
más larga, el historial se vuelve gigante y muchos de esos historiales nunca han aparecido en los
datos de entrenamiento, por lo que se hace imposible estimar las probabilidades. El segundo es el
problema de la dispersión.

Lo que hace el supuesto de Markov es resolver esto rompiendo la dependencia del historial largo,
asumiendo que la memoria del modelo es corta y limitada a los últimos k elementos. El resultado son
los modelos de bigrama y trigrama: el bigrama asume que la palabra depende únicamente de la palabra
inmediata anterior, y el trigrama de las dos anteriores.

Entonces, gracias a esto, en lugar de buscar frecuencias de frases enteras de veinte palabras, solo
buscamos combinaciones muy cortas.

## 3. Suavizado

In [ ]:
V = len(vocab_train) + 1
print(f"V = {V:,}")


def p_bigrama_addk(palabra, anterior, k=1.0):
    """P_add-k(w_n | w_n-1) = (C(w_n-1,w_n) + k) / (C(w_n-1) + k*V)"""
    numerador = conteo_bi[(anterior, palabra)] + k
    denominador = total_ctx_bi[anterior] + k * V
    return numerador / denominador if denominador > 0 else 0.0

suma = sum(p_bigrama_addk(w, "don") for w in vocab_train) + p_bigrama_addk("<OOV>", "don")
print(f"suma de P(w | 'don') sobre todo V = {suma:.6f}   (debe dar 1.0)")

V = 22,651
suma de P(w | 'don') sobre todo V = 1.000000   (debe dar 1.0)


In [ ]:
def prob_oracion_addk(oracion, k=1.0):
    """Log-probabilidad de una oracion bajo el bigrama suavizado.
    Ya no hace falta cortar en cero: ningun n-grama tiene probabilidad nula."""
    o = con_padding(oracion, 2)
    log_p = 0.0
    for i in range(1, len(o)):
        p = p_bigrama_addk(o[i], o[i - 1], k)
        log_p += math.log(p)
    return log_p


_, log_sin = prob_oracion(ejemplo, "bi")
p_sin = math.exp(log_sin) if log_sin != float("-inf") else 0.0

print("Oracion:", " ".join(ejemplo), "\n")
print(f"{'modelo':<24} {'log P':>12}   {'P':>12}")
print(f"{'bigrama sin suavizar':<24} {log_sin:>12.3f}   {p_sin:>12.3e}")

for k in [1.0, 0.1, 0.01]:
    lp = prob_oracion_addk(ejemplo, k)
    etiqueta = "Laplace (k=1)" if k == 1.0 else f"add-k (k={k})"
    print(f"{etiqueta:<24} {lp:>12.3f}   {math.exp(lp):>12.3e}")

Oracion: <s> ¿Qué es esto que oigo ? </s> 

modelo                          log P              P
bigrama sin suavizar             -inf      0.000e+00
Laplace (k=1)                 -52.620      1.405e-23
add-k (k=0.1)                 -45.089      2.618e-20
add-k (k=0.01)                -41.968      5.938e-19


In [ ]:
ctx = "don"
palabra_frecuente = siguientes_don.most_common(1)[0][0]
palabra_no_vista  = "helicóptero"

print(f"'{ctx}' aparece {total_ctx_bi[ctx]:,} veces y va seguido de "
      f"{len(siguientes_don):,} palabras distintas.\n")

print(f"{'':<36} {'sin suavizar':>14} {'Laplace':>12} {'add-k 0.01':>12}")
for palabra in [palabra_frecuente, palabra_no_vista]:
    etiqueta = f"P('{palabra}' | '{ctx}')"
    print(f"{etiqueta:<36} {p_bigrama(palabra, ctx):>14.6f} "
          f"{p_bigrama_addk(palabra, ctx, 1.0):>12.6f} "
          f"{p_bigrama_addk(palabra, ctx, 0.01):>12.6f}")

'don' aparece 2,109 veces y va seguido de 60 palabras distintas.

                                       sin suavizar      Laplace   add-k 0.01
P('Quijote' | 'don')                       0.606449     0.051696     0.547636
P('helicóptero' | 'don')                   0.000000     0.000040     0.000004


**¿Qué problema resuelve el suavizado y por qué le quita probabilidad a los n-gramas frecuentes?**

El suavizado resuelve el problema de la probabilidad cero. Sin suavizado, si en una frase aparece
un n-grama que no estaba en el entrenamiento, la probabilidad de toda la frase se va a cero y con
ella todo el sentido de la oración, por más que el resto esté bien estimado. Por eso en la sección
anterior el logaritmo de la oración daba menos infinito.

Lo que hace el suavizado es crear una bolsa de probabilidad: extrae una pequeña fracción de la
probabilidad del total de los n-gramas que sí se vieron en el entrenamiento. Esa masa de
probabilidad acumulada se redistribuye después equitativamente, o bajo ciertos criterios, entre
todos los n-gramas que tienen frecuencia cero.

Entonces, con el suavizado se corta una pequeña parte de las palabras que sí se vieron y con eso la
frase completa pasa a tener una probabilidad calculable y superior a cero. En este corpus el efecto
se ve directo: la probabilidad de Quijote después de don cae de 0.606449 a 0.051696 con
Laplace, y esa masa que pierde es la que termina repartida entre las combinaciones que nunca
aparecieron, como helicóptero después de don, que pasa de cero a 0.000040.

## 4. Evaluación con perplejidad

La perplejidad mide qué tan sorprendido queda el modelo ante texto que no vio. Es la
probabilidad de todo el conjunto, normalizada por su número de tokens e invertida:

In [ ]:
def p_unigrama_addk(palabra, k=1.0):
    return (conteo_uni[palabra] + k) / (total_uni + k * V)


def p_trigrama_addk(palabra, ant2, ant1, k=1.0):
    return (conteo_tri[(ant2, ant1, palabra)] + k) / (total_ctx_tri[(ant2, ant1)] + k * V)


def perplejidad(oraciones, modelo, k=1.0):
    pad = [con_padding(o, 3) for o in oraciones]
    n_tokens = sum(len(o) - 2 for o in pad)

    if modelo == "uni":
        log_total = sum(math.log(p_unigrama_addk(w, k))
                        for w in chain.from_iterable(o[2:] for o in pad))
    elif modelo == "bi":
        log_total = sum(math.log(p_bigrama_addk(w, ctx, k))
                        for ctx, w in chain.from_iterable(zip(o[1:], o[2:]) for o in pad))
    else:
        log_total = sum(math.log(p_trigrama_addk(w, a, b, k))
                        for a, b, w in chain.from_iterable(zip(o, o[1:], o[2:]) for o in pad))

    return math.exp(-log_total / n_tokens)


filas = [{"modelo": nombre, "perplejidad (val)": round(perplejidad(val_oraciones, m), 2)}
         for nombre, m in [("unigrama", "uni"), ("bigrama", "bi"), ("trigrama", "tri")]]
tabla_pp = pd.DataFrame(filas).set_index("modelo")
tabla_pp

,perplejidad (val)
modelo,
unigrama,624.03
bigrama,2094.39
trigrama,10319.14


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
tabla_pp["perplejidad (val)"].plot(kind="bar", ax=ax, color=["steelblue", "seagreen", "indianred"])
ax.set_ylabel("Perplejidad (menor = mejor)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=.3)
for i, v in enumerate(tabla_pp["perplejidad (val)"]):
    ax.text(i, v, f"{v:,.0f}", ha="center", va="bottom")
ax.set_title("Perplejidad sobre validacion (suavizado Laplace, k=1)")
plt.tight_layout()
plt.show()

<Figure size 700x400 with 1 Axes>

In [ ]:
ks = [1.0, 0.5, 0.1, 0.05, 0.01, 0.005, 0.001]

tabla_k = pd.DataFrame(
    {nombre: [round(perplejidad(val_oraciones, m, k), 2) for k in ks]
     for nombre, m in [("unigrama", "uni"), ("bigrama", "bi"), ("trigrama", "tri")]},
    index=ks,
)
tabla_k.index.name = "k"

fig, ax = plt.subplots(figsize=(8, 4.5))
for col, color in zip(tabla_k.columns, ["steelblue", "seagreen", "indianred"]):
    ax.plot(tabla_k.index, tabla_k[col], marker="o", label=col, color=color)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("k (escala log)")
ax.set_ylabel("Perplejidad (escala log)")
ax.set_title("Perplejidad sobre validacion segun el suavizado")
ax.legend(); ax.grid(alpha=.3, which="both")
plt.tight_layout()
plt.show()

tabla_k

<Figure size 800x450 with 1 Axes>

,unigrama,bigrama,trigrama
k,,,
1.000,624.03,2094.39,10319.14
0.500,630.33,1509.03,8412.90
0.100,659.82,804.32,5093.41
0.050,674.95,659.79,4148.20
0.010,712.52,495.04,2790.16
0.005,729.50,469.28,2469.39
0.001,770.63,479.00,2154.05


In [ ]:
mejor_modelo = tabla_k.min().idxmin()          
mejor_k      = tabla_k[mejor_modelo].idxmin()  
clave = {"unigrama": "uni", "bigrama": "bi", "trigrama": "tri"}[mejor_modelo]

print("Mejor de cada modelo sobre validacion:")
for col in tabla_k.columns:
    print(f"  {col:<10} PP = {tabla_k[col].min():>9,.2f}  con k = {tabla_k[col].idxmin()}")

print(f"\nGanador: {mejor_modelo} con k = {mejor_k}")
print(f"Perplejidad final sobre PRUEBA: {perplejidad(test_oraciones, clave, mejor_k):,.2f}")

Mejor de cada modelo sobre validacion:
  unigrama   PP =    624.03  con k = 1.0
  bigrama    PP =    469.28  con k = 0.005
  trigrama   PP =  2,154.05  con k = 0.001

Ganador: bigrama con k = 0.005
Perplejidad final sobre PRUEBA: 475.55


**¿Cuál modelo obtiene menor perplejidad? ¿Coincide con el trade-off entre contexto y dispersión? ¿Cómo cambia la perplejidad al aumentar k?**

Con suavizado de Laplace el que gana es el unigrama, con 624.03 de perplejidad sobre validación,
mientras que el bigrama queda en 2,094.39 y el trigrama en 10,319.14. Ese orden es exactamente el
contrario al que uno esperaría, pero no significa que tener menos contexto sea mejor. Lo que pasa es
que k=1 reparte 22,651 conteos ficticios en cada contexto, y los contextos de bigrama y de trigrama
tienen muy pocas observaciones reales para defenderse de esa cantidad. El suavizado los ahoga.

Al ajustar k el resultado cambia por completo y ahí sí aparece el trade-off. El bigrama baja hasta
469.28 con k=0.005 y pasa a ser el mejor modelo, por debajo del unigrama, que en su mejor caso se
queda en 624.03. Es decir que agregar una palabra de contexto sí ayuda. El trigrama en cambio sigue
siendo el peor por mucho incluso en su mejor configuración, con 2,154.05, porque sus 129,305
contextos distintos son demasiado dispersos y casi ninguno tiene suficientes observaciones para
estimar bien. Ganamos al pasar de una a dos palabras de contexto y perdemos al pasar de dos a tres,
así que el punto óptimo está en el medio, que es justo lo que dice la teoría.

Sobre el efecto de k, la perplejidad del bigrama sube conforme k crece: pasa de 469.28 con k=0.005
a 495.04 con k=0.01, a 804.32 con k=0.1 y a 2,094.39 con k=1. La razón es que un k más grande le
quita cada vez más masa de probabilidad a los bigramas que sí se observaron para dársela a los que
nunca aparecieron, y el modelo se vuelve más plano y menos capaz de distinguir. Por debajo del
óptimo el efecto se invierte: con k=0.001 la perplejidad vuelve a subir a 479.00, porque el
suavizado ya no alcanza a cubrir lo no visto. Cada modelo prefiere un k distinto, y ese valor baja
mientras más disperso es el modelo: el unigrama funciona mejor con k=1, el bigrama con k=0.005 y el
trigrama con k=0.001.

El mejor par de modelo y suavizado según validación es el bigrama con k=0.005, y su perplejidad
final sobre el conjunto de prueba es de 475.55, muy cerca de los 469.28 de validación.

## 5. Aplicación práctica: autocompletado

In [ ]:
indice_bi = defaultdict(Counter)
for (ctx, palabra), c in conteo_bi.items():
    indice_bi[ctx][palabra] = c

print(f"contextos indexados: {len(indice_bi):,}")


def autocompletar(fragmento, n=5):
    """Dado un fragmento de oracion, devuelve las n palabras siguientes mas probables."""
    tokens = ["<s>"] + word_tokenize(fragmento, language="spanish")
    ctx = tokens[-1]
    total = total_ctx_bi[ctx]
    if total == 0:
        return []
    return [(w, c / total) for w, c in indice_bi[ctx].most_common(n)]


for w, prob in autocompletar("En un lugar de la"):
    print(f"  {w:<15} {prob:.4f}")

contextos indexados: 22,649
  mano            0.0182
  Mancha          0.0165
  cabeza          0.0164
  cual            0.0159
  que             0.0149


In [ ]:
random.seed(7)
largas = [o for o in test_oraciones if len(o) >= 12]
muestra = random.sample(largas, 5)

for oracion in muestra:
    corte = len(oracion) // 2
    fragmento = " ".join(oracion[1:corte])          
    real = oracion[corte]                            

    print(f"fragmento : ...{' '.join(oracion[max(1, corte-6):corte])}")
    print(f"palabra real siguiente: {real!r}")
    sugerencias = autocompletar(fragmento)
    if sugerencias:
        print("  sugerencias:", ", ".join(f"{w} ({pr:.3f})" for w, pr in sugerencias))
        print(f"  la real esta entre las 5: {real in [w for w, _ in sugerencias]}")
    else:
        print("  (contexto no visto en entrenamiento)")
    print()

fragmento : ...no tiene hijo ninguno , ni
palabra real siguiente: 'varón'
  sugerencias: de (0.036), en (0.034), a (0.033), más (0.033), aun (0.025)
  la real esta entre las 5: False

fragmento : ...sabía yo por muy cierto que
palabra real siguiente: 'era'
  sugerencias: no (0.061), , (0.052), se (0.044), le (0.033), el (0.030)
  la real esta entre las 5: False

fragmento : ...aguileña y la nariz algo chata
palabra real siguiente: ','
  sugerencias: y (0.333), , (0.333), ; (0.333)
  la real esta entre las 5: True

fragmento : ...digáis a don Quijote quién soy
palabra real siguiente: ','
  sugerencias: , (0.095), yo (0.091), el (0.076), de (0.072), don (0.027)
  la real esta entre las 5: True

fragmento : ...-dijo Sancho- , excepto aquello de
palabra real siguiente: 'la'
  sugerencias: la (0.114), los (0.053), su (0.050), las (0.035), mi (0.023)
  la real esta entre las 5: True



**¿Son razonables las sugerencias? ¿Se nota que el corpus es del siglo XVII?**

Las sugerencias son razonables en general. De los cinco fragmentos tomados de oraciones reales del
conjunto de prueba, en tres el modelo puso la palabra correcta entre sus cinco propuestas. En
*aguileña y la nariz algo chata* la palabra real era una coma y el modelo propuso coma, punto y coma
y la conjunción *y*, que son las tres continuaciones posibles en ese punto. En *digáis a don Quijote
quién soy* la coma real aparece como primera sugerencia, y en *excepto aquello de* el modelo acierta
con *la* en primer lugar, seguida de *los*, *su*, *las* y *mi*, que son todas continuaciones válidas.

En los dos casos donde falla, la palabra real igual tiene todo el sentido. En *no tiene hijo ninguno,
ni* la palabra que seguía era *varón*, que encaja perfectamente, pero el modelo propuso preposiciones
como *de*, *en* y *a*. Lo mismo en *sabía yo por muy cierto que*, donde seguía *era* y el modelo
sugirió *no*, una coma y *se*. El patrón es claro: cuando el contexto es una palabra funcional como
*ni* o *que*, el bigrama solo puede ofrecer las palabras que más suelen seguirlas en general, que
son otras palabras funcionales, porque una sola palabra de contexto no alcanza para saber de qué
trata la frase.

Y sí se nota que el corpus es del siglo XVII. Al pedir la continuación de *En un lugar de la* el
modelo propone *mano*, *Mancha*, *cabeza*, *cual* y *que*, con *Mancha* en segundo lugar, que es
una palabra que ningún corpus de español actual pondría ahí. Las sugerencias en general están
formadas por vocabulario y giros de la época, y el modelo no conoce ninguna palabra moderna porque
simplemente no existe en el texto del que aprendió.